# ARC-AGI Crew — Kaggle Notebook (Gemma 4 E4B local)

## 1. Install dependencies

`kagglehub` and `transformers` are pre-installed on Kaggle.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links=/kaggle/input/datasets/yusuftryak/offwhl/kaggle/working/updated_libs",
    "--upgrade", "transformers", "accelerate", "torchvision", "torchaudio",
], check=True)
print("Kütüphaneler güncellendi (offline).")

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links=/kaggle/input/datasets/yusuftryak/offwhl/crew/kaggle/working/crew",
    "uvicorn==0.46.0", "crewai==1.14.4", "onnx", "onnxruntime", "onnx-tool",
], check=True)
print("Crew kütüphaneleri kuruldu (offline).")

## 2. Load Gemma 4 E4B

Loaded once; shared by all three agent LLM instances.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

_MODEL_ID = "gemma-4-e4b-it"
_GEMMA_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

_processor = AutoProcessor.from_pretrained(_GEMMA_PATH)
_model = AutoModelForCausalLM.from_pretrained(
    _GEMMA_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)
_model.eval()
print("Loaded on:", next(_model.parameters()).device)


## 3. GemmaLocalLLM — crewai BaseLLM wrapper

In [ ]:
import re as _re_lib
from typing import Any
from pydantic import PrivateAttr
from crewai.llms.base_llm import BaseLLM

_THINK_RE = _re_lib.compile(r"<think>.*?</think>", _re_lib.DOTALL)


class GemmaLocalLLM(BaseLLM):
    """crewai BaseLLM subclass wrapping the locally-loaded Gemma 4 E4B model.

    The model and processor globals (_model, _processor) are bound via PrivateAttr
    so they are shared across all three agent instances without reloading.
    crewai uses the ReAct text path because supports_function_calling() is absent.
    """

    _gemma_model: Any = PrivateAttr(default=None)
    _gemma_processor: Any = PrivateAttr(default=None)

    def __init__(self, temperature: float = 0.2, **kwargs: Any) -> None:
        super().__init__(model=_MODEL_ID, temperature=temperature, **kwargs)
        # Bind already-loaded globals — no reloading, zero extra VRAM.
        object.__setattr__(self, "_gemma_model", _model)
        object.__setattr__(self, "_gemma_processor", _processor)

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
    ) -> str:
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        text = self._gemma_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
            thinking_budget=512,
        )
        # Wrap text in a list — required by newer transformers processor API.
        inputs = self._gemma_processor(text=[text], return_tensors="pt")
        first_device = next(self._gemma_model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[-1]

        torch.cuda.empty_cache()
        with torch.no_grad():
            output_ids = self._gemma_model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=self.temperature if self.temperature and self.temperature > 0 else 1.0,
                do_sample=bool(self.temperature and self.temperature > 0),
            )

        raw = self._gemma_processor.decode(
            output_ids[0][input_len:],
            skip_special_tokens=False,
        )

        # Free GPU memory immediately after decoding.
        del inputs, output_ids
        torch.cuda.empty_cache()

        try:
            parsed = self._gemma_processor.parse_response(raw)
            result = parsed.get("response", raw)
        except Exception:
            result = _THINK_RE.sub("", raw).strip()

        # Honour crewai's "\nObservation:" stop word (injected per-agent by executor).
        return self._apply_stop_words(result)

    def get_context_window_size(self) -> int:
        return 131072  # Gemma 4 128k context


## 3b. Smoke test

In [ ]:
_test_llm = GemmaLocalLLM(temperature=0.1)
_test_llm.stop = ["\nObservation:"]
_test_out = _test_llm.call([{"role": "user", "content": "Reply with exactly: Final Answer: OK"}])
print(repr(_test_out))
assert "OK" in _test_out or "Final" in _test_out, f"Unexpected output: {_test_out!r}"
print("GemmaLocalLLM smoke test PASSED")


## 4. neurogolf_utils (verbatim)

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Module containing utilities for the IJCAI-ECAI 2026 NeuroGolf Championship.

Version History:
* 2026-05-06:
    * Scalar parameters are now penalized with unit cost.
    * Each tensor's memory footprint is set to the maximum size across all runs.
    * Duplicate node names no longer create parameter undercount.
    * Tensor names containing ONNX's special "kernel_time" string are disallowed.
    * Runtime trace file prefixes are specified to prevent profile clobbering.
    * Multi-input / multi-output graphs disallowed.
* 2026-05-04:
    * Sequences and nonpositive tensor dimensions are disallowed.
    * Accurate shape information derived from the ONNX Runtime Profiler.
    * MACs no longer contribute to the objective criterion.
* 2026-04-30:
    * Compress operators have been banned.
    * Name collision between tensors and initializers are disallowed.
    * Functions / custom domains / subgraphs are disallowed.
    * Zero-cost networks now yield a full 25 points.
* 2026-04-28:
    * Constant folding enabled to address the undercounting of parameters.
    * Our "statically-defined shapes" constaint is now strictly enforced.
    * Memory footprint calculation is now a sum of static shape sizes.
    * Nodes with negative parameter counts or MACs are disallowed.
* 2026-04-21:
    * Tests with grids larger than 30x30 are ignored.
    * Nodes with negative memory values are disallowed.
* 2026-04-15:
    * Initial version.

Contributors from the Kaggle Community:
* @anglolodorf
* @arc144
* @asalhi
* @calibrator
* @cdeotte
* @hengck23
* @jazivxt
* @jiweiliu
* @kameronkilchrist
* @kevinyuluo
* @kosirowada
* @maxjeblick
* @mukundan314
* @pavelsavchenkov
* @prokaj
* @robga
* @shinh0
* @tonylica
* @yeoyunsianggeremie
* @yiheng
"""

import itertools
import json
import math
import pathlib
import traceback

import IPython.display
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnx_tool
import onnxruntime


display = IPython.display.display
FileLink = IPython.display.FileLink

_BATCH_SIZE, _CHANNELS, _HEIGHT, _WIDTH = 1, 10, 30, 30
_NEUROGOLF_DIR = "/kaggle/input/datasets/yusuftryak/offwhl/arc-agi-1/arc-agi-1/"
_COLORS = [
    (0, 0, 0),
    (30, 147, 255),
    (250, 61, 49),
    (78, 204, 48),
    (255, 221, 0),
    (153, 153, 153),
    (229, 59, 163),
    (255, 133, 28),
    (136, 216, 241),
    (147, 17, 49),
    (240, 240, 240),
    (146, 117, 86)
]
_DATA_TYPE = onnx.TensorProto.FLOAT
_EXCLUDED_OP_TYPES = ["LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"]
_FILESIZE_LIMIT_IN_BYTES = 1.44 * 1024 * 1024
_GRID_SHAPE = [_BATCH_SIZE, _CHANNELS, _HEIGHT, _WIDTH]
_IR_VERSION, _OPSET_IMPORTS = 10, [onnx.helper.make_opsetid("", 10)]
_TASK_ZERO = {
    "train": [{
        "input": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        ],
        "output": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 5, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 0, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 0, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 0, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 0, 5, 5],
            [5, 1, 1, 1, 1, 1, 1, 0, 5, 5],
            [5, 5, 0, 0, 0, 0, 0, 0, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        ],
    }],
    "test": [{
        "input": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 4, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 4, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 5, 5, 5],
            [5, 5, 4, 5, 5, 5, 4, 5, 5, 5],
            [5, 5, 4, 5, 5, 5, 4, 5, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        ],
        "output": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 4, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 4, 0, 5],
            [5, 5, 5, 0, 0, 0, 0, 0, 0, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 5, 5, 5],
            [5, 5, 4, 0, 0, 0, 4, 0, 5, 5],
            [5, 5, 4, 0, 5, 5, 4, 0, 5, 5],
            [5, 5, 4, 4, 4, 4, 4, 0, 5, 5],
            [5, 5, 5, 0, 0, 0, 0, 0, 5, 5],
        ],
    }],
    "arc-gen": [{
        "input": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 2, 2, 2, 2, 2, 2, 5, 5],
            [5, 5, 2, 5, 5, 5, 5, 2, 5, 5],
            [5, 5, 2, 5, 5, 5, 5, 2, 5, 5],
            [5, 5, 2, 5, 5, 5, 5, 2, 5, 5],
            [5, 5, 2, 5, 5, 5, 5, 2, 5, 5],
            [5, 5, 2, 2, 2, 2, 2, 2, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        ],
        "output": [
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
            [5, 5, 2, 2, 2, 2, 2, 2, 5, 5],
            [5, 5, 2, 0, 0, 0, 0, 2, 0, 5],
            [5, 5, 2, 0, 5, 5, 5, 2, 0, 5],
            [5, 5, 2, 0, 5, 5, 5, 2, 0, 5],
            [5, 5, 2, 0, 5, 5, 5, 2, 0, 5],
            [5, 5, 2, 2, 2, 2, 2, 2, 0, 5],
            [5, 5, 5, 0, 0, 0, 0, 0, 0, 5],
            [5, 5, 5, 5, 5, 5, 5, 5, 5, 5],
        ],
    }],
}


def calculate_memory(model, trace_path):
    onnx.checker.check_model(model, full_check=True)
    graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph
    if len(graph.input) > 1 or len(graph.output) > 1: return None
    init_names = {init.name for init in graph.initializer}
    init_names.update(init.name for init in graph.sparse_initializer)
    io_names = {t.name for t in list(graph.input) + list(graph.output)}
    if io_names.intersection(init_names): return None
    if model.functions: return None
    for opset in model.opset_import:
        if opset.domain not in {"", "ai.onnx"}: return None
    node_outputs = {}
    tensor_names = set()
    for node in graph.node:
        for attr in node.attribute:
            if attr.type in [onnx.AttributeProto.GRAPH,
                             onnx.AttributeProto.GRAPHS]:
                return None
        node_outputs[node.name] = list(node.output)
        for output_name in node.output:
            if output_name: tensor_names.add(output_name)
    tensor_memory = {}
    tensor_dtypes = {}
    tensor_map = {
        t.name: t for t in list(graph.input) + list(graph.value_info) + list(graph.output)
    }
    tensor_names.update(tensor_map.keys())
    for tensor_name in tensor_names:
        item = tensor_map.get(tensor_name)
        if not item: return None
        if item.type.HasField("sequence_type"): return None
        if not item.type.HasField("tensor_type"): continue
        tensor_type = item.type.tensor_type
        if not tensor_type.HasField("shape"): return None
        num_elements = 1
        for dim in tensor_type.shape.dim:
            if dim.HasField("dim_param"): return None
            if not dim.HasField("dim_value"): return None
            if dim.dim_value <= 0: return None
            num_elements *= dim.dim_value
        if tensor_name in ['input', 'output']: continue
        np_dtype = onnx.helper.tensor_dtype_to_np_dtype(tensor_type.elem_type)
        tensor_memory[tensor_name] = num_elements * np.dtype(np_dtype).itemsize
        tensor_dtypes[tensor_name] = np_dtype

    # Retrieve actual tensor shapes via the ONNX Runtime Profiler's JSON Trace.
    with open(trace_path, 'r') as f:
        trace_data = json.load(f)
    for event in trace_data:
        if event.get("cat") != "Node" or "args" not in event: continue
        if "output_type_shape" not in event["args"]: continue
        node_name = event.get("name").replace("_kernel_time", "")
        if node_name not in node_outputs: continue
        for i, shape_dict in enumerate(event["args"]["output_type_shape"]):
            if i >= len(node_outputs[node_name]): continue
            output_name = node_outputs[node_name][i]
            if output_name not in tensor_dtypes: continue
            itemsize = np.dtype(tensor_dtypes[output_name]).itemsize
            mem = itemsize * sum(math.prod(dims) for dims in shape_dict.values())
            tensor_memory[output_name] = max(tensor_memory[output_name], mem)
    return sum(tensor_memory.values())

def check_network(filename):
  file_path = pathlib.Path(filename)
  if not file_path.is_file():
    print(f"Error: File {filename} does not exist.")
    return False
  if (filesize := file_path.stat().st_size) > _FILESIZE_LIMIT_IN_BYTES:
    print(f"Error: Filesize {filesize} exceeds {_FILESIZE_LIMIT_IN_BYTES}.")
    return False
  return True


def convert_to_numpy(example):
  benchmark = {}
  example_shape = (1, _CHANNELS, _HEIGHT, _WIDTH)
  for mode in ["input", "output"]:
    benchmark[mode] = np.zeros(example_shape, dtype=np.float32)
    grid = example[mode]
    if max(len(grid), len(grid[0])) > 30: return None
    for r, _ in enumerate(grid):
      for c, color in enumerate(grid[r]):
        benchmark[mode][0][color][r][c] = 1.0
  return benchmark


def convert_from_numpy(benchmark):
  example = []
  _, channels, height, width = benchmark.shape
  for row in range(height):
    cells = []
    for col in range(width):
      colors = [c for c in range(channels) if benchmark[0][c][row][col] == 1]
      cells.append(colors[0] if len(colors) == 1 else (11 if colors else 10))
    while cells and cells[-1] == 10:
      cells.pop(-1)
    example.append(cells)
  while example and not example[-1]:
    example.pop(-1)
  return example


def calculate_params(model):
    params = 0
    for init in model.graph.initializer:
        if any(d <= 0 for d in init.dims): return None
        params += math.prod(init.dims)
    for sparse_init in model.graph.sparse_initializer:
        if any(d <= 0 for d in sparse_init.values.dims): return None
        params += math.prod(sparse_init.values.dims)
    for node in model.graph.node:
        if node.op_type != 'Constant': continue
        for attr in node.attribute:
            if attr.name == 'value':
                if any(d <= 0 for d in attr.t.dims): return None
                params += math.prod(attr.t.dims)
            elif attr.name == 'sparse_value':
                if any(d <= 0 for d in attr.sparse_tensor.values.dims): return None
                params += math.prod(attr.sparse_tensor.values.dims)
            elif attr.name == 'value_floats':
                params += len(attr.floats)
            elif attr.name == 'value_ints':
                params += len(attr.ints)
            elif attr.name == 'value_strings':
                params += len(attr.strings)
    return params


def score_network(sanitized, trace_path):
    return calculate_memory(sanitized, trace_path), calculate_params(sanitized)


def load_examples(task_num):
  """Loads relevant data from ARC-AGI and ARC-GEN."""
  if not task_num:
    return _TASK_ZERO
  with open(_NEUROGOLF_DIR + f"task{task_num:03d}.json") as f:
    examples = json.load(f)
  return examples


def run_network(session, benchmark_input):
  result = session.run(["output"], {"input": benchmark_input})
  return (result[0] > 0.0).astype(float)


def show_examples(examples, bgcolor=(255, 255, 255)):
  # Determine the dimensions of the image to be rendered.
  width, height, offset = 0, 0, 1
  for example in examples:
    grid, output = example["input"], example["output"]
    width += len(grid[0]) + 1 + len(output[0]) + 4
    height = max(height, max(len(grid), len(output)) + 4)
  # Determine the contents of the image.
  image = [[bgcolor for _ in range(width)] for _ in range(height)]
  for example in examples:
    grid, output = example["input"], example["output"]
    grid_width, output_width = len(grid[0]), len(output[0])
    for r, row in enumerate(grid):
      for c, cell in enumerate(row):
        image[r + 2][offset + c + 1] = _COLORS[cell]
    offset += grid_width + 1
    for r, row in enumerate(output):
      for c, cell in enumerate(row):
        image[r + 2][offset + c + 1] = _COLORS[cell]
    offset += output_width + 4
  # Draw the image.
  fig = plt.figure(figsize=(10, 5))
  ax = fig.add_axes([0, 0, 1, 1])
  ax.imshow(np.array(image))
  # Draw the horizontal and vertical lines.
  offset = 1
  for example in examples:
    grid, output = example["input"], example["output"]
    grid_width, grid_height = len(grid[0]), len(grid)
    output_width, output_height = len(output[0]), len(output)
    ax.hlines([r + 1.5 for r in range(grid_height+1)],
              xmin=offset+0.5, xmax=offset+grid_width+0.5, color="black")
    ax.vlines([offset + c + 0.5 for c in range(grid_width+1)],
              ymin=1.5, ymax=grid_height+1.5, color="black")
    offset += grid_width + 1
    ax.hlines([r + 1.5 for r in range(output_height+1)],
              xmin=offset+0.5, xmax=offset+output_width+0.5, color="black")
    ax.vlines([offset + c + 0.5 for c in range(output_width+1)],
              ymin=1.5, ymax=output_height+1.5, color="black")
    offset += output_width + 2
    ax.vlines([offset+0.5], ymin=-0.5, ymax=height-0.5, color="black")
    offset += 2
  ax.set_xticks([])
  ax.set_yticks([])


def show_legend():
  image = [[(255, 255, 255) for _ in range(21)] for _ in range(5)]
  for idx, color in enumerate(_COLORS[:10]):
    image[1][2 * idx + 1] = color
  for idx, color in enumerate(_COLORS[10:]):
    for col in range(3):
      image[3][12 * idx + col + 3] = color
  fig = plt.figure(figsize=(10, 5))
  ax = fig.add_axes([0, 0, 1, 1])
  ax.imshow(np.array(image))
  for idx, _ in enumerate(_COLORS[:10]):
    color = "white" if idx in [0, 9] else "black"
    ax.text(2 * idx + 0.9, 1.1, str(idx), color=color)
  ax.text(3.4, 3.1, "no color", color="black")
  ax.text(5.75, 3.1, "<--- special colors to indicate one-hot encoding errors --->", color="black")
  ax.text(14.85, 3.1, "too many colors", color="white")
  ax.set_xticks([])
  ax.set_yticks([])


def single_layer_conv2d_network(weight_fn, kernel_size):
  kernel_offsets = range(-kernel_size // 2 + 1, kernel_size // 2 + 1)
  kernel_shape = [kernel_size, kernel_size]
  w_shape = [_CHANNELS, _CHANNELS, kernel_size, kernel_size]
  pads = [kernel_size // 2] * 4
  weight_cells = itertools.product(range(_CHANNELS), range(_CHANNELS),
                                   kernel_offsets, kernel_offsets)
  weights = [weight_fn(o, i, (r, c)) for (o, i, r, c) in weight_cells]

  x = onnx.helper.make_tensor_value_info("input", _DATA_TYPE, _GRID_SHAPE)
  y = onnx.helper.make_tensor_value_info("output", _DATA_TYPE, _GRID_SHAPE)
  w = onnx.helper.make_tensor("W", _DATA_TYPE, w_shape, weights)
  node_def = onnx.helper.make_node("Conv", ["input", "W"], ["output"],
                                   kernel_shape=kernel_shape, pads=pads)
  graph_def = onnx.helper.make_graph([node_def], "graph", [x], [y], [w])
  model_def = onnx.helper.make_model(graph_def, ir_version=_IR_VERSION,
                                     opset_imports=_OPSET_IMPORTS)
  return model_def


def verify_network(network, task_num, examples):
  filename = "task{:03d}.onnx".format(task_num)
  onnx.save(network, filename)
  if not check_network(filename): return
  try:
    # Load the model, sanitize node names, and enable profiling.
    sanitized = onnx.load(filename)
    for node in sanitized.graph.node:
        node.name = node.output[0]
        if "kernel_time" in node.name: return
    options = onnxruntime.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
    options.profile_file_prefix = f"{task_num:03}"
    session = onnxruntime.InferenceSession(sanitized.SerializeToString(), options)
  except onnxruntime.ONNXRuntimeError as e:
    print(f"Error: Unable to load ONNX model: {e}")
    return
  arc_agi_right, arc_agi_wrong, arc_agi_expected = verify_subset(session, examples["train"] + examples["test"])
  arc_gen_right, arc_gen_wrong, arc_gen_expected = verify_subset(session, examples["arc-gen"])
  print(f"Results on ARC-AGI examples: {arc_agi_right} pass, {arc_agi_wrong} fail")
  print(f"Results on ARC-GEN examples: {arc_gen_right} pass, {arc_gen_wrong} fail")
  print()
  memory, params = score_network(sanitized, session.end_profiling())
  if memory is None or params is None:
    print("Error: Your network performance could not be measured")
  if memory < 0 or params < 0:
    print("Error: Your network performance could not be measured")
  elif arc_agi_wrong + arc_gen_wrong == 0:
    print("Your network IS READY for submission!")
    print()
    print("Performance stats (memory values reported here are approximate):")
    onnx_tool.model_profile(filename)
    points = max(1.0, 25.0 - math.log(max(1.0, memory + params)))
    print()
    print(f"It appears to require {memory} bytes + {params} params, yielding {points:.3f} points.")
    print()
    print("Next steps:")
    print(f" * Click the link below to download {filename} onto your local machine.")
    print(" * Create a zip file containing that network along with all others.")
    print(" * Submit that zip file to the Kaggle competition so that it can be officially scored.")
    print()
    display(FileLink(filename))
  else:
    print("Your network IS NOT ready for submission.")
    expected = None
    expected = arc_agi_expected if arc_agi_expected is not None else expected
    expected = arc_gen_expected if arc_gen_expected is not None else expected
    if expected is None: return
    benchmark = convert_to_numpy(expected)
    actual = {}
    actual["input"] = expected["input"]
    actual["output"] = convert_from_numpy(run_network(session, benchmark["input"]))
    print("The expected result is shown in green; your actual result is shown in red.")
    show_examples([expected], bgcolor=(200, 255, 200))
    show_examples([actual], bgcolor=(255, 200, 200))


def verify_subset(session, example_subset):
  right, wrong, expected, error = 0, 0, None, ""
  for example in example_subset:
    benchmark = convert_to_numpy(example)
    if not benchmark: continue
    try:
      user_output = run_network(session, benchmark["input"])
      if np.array_equal(user_output, benchmark["output"]):
        right += 1
      else:
        expected = example
        wrong += 1
    except onnxruntime.ONNXRuntimeError:
      error = traceback.format_exc()
      wrong += 1
  if error: print(f"Error: {error}")
  return right, wrong, expected

## 5. Tools

In [ ]:
import json
import math
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import scipy.ndimage as ndi

import onnx
import onnxruntime
from crewai.tools import BaseTool

_SCRATCH_DIR = Path("/tmp/arc_agi_scratch")
_KAGGLE_TASK_DIR = Path("/kaggle/input/datasets/yusuftryak/offwhl/arc-agi-1/arc-agi-1")
_LOCAL_TASK_DIR = Path("tasks")

def _load_task(task_num: int) -> dict:
    """Load task JSON, trying Kaggle path first then local fallback."""
    if task_num == 0:
        return load_examples(0)
    for path in (
        _KAGGLE_TASK_DIR / f"task{task_num:03d}.json",
        _LOCAL_TASK_DIR / f"task{task_num:03d}.json",
    ):
        if path.exists():
            with open(path) as f:
                return json.load(f)
    return load_examples(task_num)

class GridStatsTool(BaseTool):
    name: str = "grid_stats"
    description: str = (
        "Extracts structured statistics for an ARC-AGI task. "
        "Call this FIRST — before forming any strategy. "
        "Input: task_num (int) — the integer task number (use 0 for the built-in test task). "
        "Do NOT read any files yourself; this tool loads the task internally. "
        "Returns a JSON string with bounding boxes, translation vectors, symmetry tests, "
        "cross-example color invariants, and per-color histograms."
    )

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _grid_to_array(grid: list) -> np.ndarray:
        """Convert a ragged list-of-lists grid to a padded int32 numpy array."""
        h = len(grid)
        w = max(len(r) for r in grid) if grid else 0
        arr = np.zeros((h, w), dtype=np.int32)
        for i, row in enumerate(grid):
            arr[i, : len(row)] = row
        return arr

    @staticmethod
    def _bounding_boxes(arr: np.ndarray, color: int) -> list:
        """Return list of {label, ymin, ymax, xmin, xmax, area} for each connected component of `color`."""
        mask = arr == color
        labeled, n = ndi.label(mask)
        all_slices = ndi.find_objects(labeled)
        boxes = []
        for lbl in range(1, n + 1):
            slices = all_slices[lbl - 1]
            if slices is None:
                continue
            ys, xs = slices
            area = int(np.sum(labeled[ys, xs] == lbl))
            boxes.append({
                "label": lbl,
                "ymin": int(ys.start),
                "ymax": int(ys.stop - 1),
                "xmin": int(xs.start),
                "xmax": int(xs.stop - 1),
                "area": area,
            })
        return boxes

    @staticmethod
    def _translation_vector(in_arr: np.ndarray, out_arr: np.ndarray, color: int):
        h = max(in_arr.shape[0], out_arr.shape[0])
        w = max(in_arr.shape[1], out_arr.shape[1])

        def pad(a):
            out = np.zeros((h, w), dtype=np.float32)
            out[: a.shape[0], : a.shape[1]] = a
            return out

        m_in = pad((in_arr == color).astype(np.float32))
        m_out = pad((out_arr == color).astype(np.float32))

        if m_in.sum() == 0 or m_out.sum() == 0:
            return None

        F_in = np.fft.fft2(m_in)
        F_out = np.fft.fft2(m_out)
        cross = np.fft.ifft2(F_out * np.conj(F_in)).real
        peak = np.unravel_index(np.argmax(cross), cross.shape)
        dy = int(peak[0]) if peak[0] <= h // 2 else int(peak[0]) - h
        dx = int(peak[1]) if peak[1] <= w // 2 else int(peak[1]) - w
        return [dy, dx]

    @staticmethod
    def _symmetry_tests(in_arr: np.ndarray, out_arr: np.ndarray) -> dict:
        if in_arr.shape != out_arr.shape:
            return {"flipud": False, "fliplr": False, "rot90_1": False, "rot90_2": False, "rot90_3": False}
        return {
            "flipud":  bool(np.array_equal(np.flipud(in_arr), out_arr)),
            "fliplr":  bool(np.array_equal(np.fliplr(in_arr), out_arr)),
            "rot90_1": bool(np.array_equal(np.rot90(in_arr, 1), out_arr)),
            "rot90_2": bool(np.array_equal(np.rot90(in_arr, 2), out_arr)),
            "rot90_3": bool(np.array_equal(np.rot90(in_arr, 3), out_arr)),
        }

    @staticmethod
    def _histograms(arr: np.ndarray, color: int) -> dict:
        mask = (arr == color).astype(int)
        return {
            "row_sums": mask.sum(axis=1).tolist(),
            "col_sums": mask.sum(axis=0).tolist(),
        }

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------

    def _run(self, task_num: int) -> str:
        try:
            task = _load_task(int(task_num))
        except Exception as e:
            return f"Task load error for task_num={task_num}: {e}"

        stats: dict = {"examples": []}
        invariant_input_colors: set | None = None
        invariant_output_colors: set | None = None
        shape_ratios: list = []

        static_input_masks: dict[int, np.ndarray | None] = {}
        static_output_masks: dict[int, np.ndarray | None] = {}
        static_input_unchanged: dict[int, bool] = {}
        static_output_unchanged: dict[int, bool] = {}

        for split in ("train", "test"):
            for ex in task.get(split, []):
                inp_raw = ex.get("input", [])
                out_raw = ex.get("output", [])
                if not inp_raw or not out_raw:
                    continue

                in_arr = self._grid_to_array(inp_raw)
                out_arr = self._grid_to_array(out_raw)
                h_in, w_in = in_arr.shape
                h_out, w_out = out_arr.shape

                in_colors = set(int(v) for v in np.unique(in_arr))
                out_colors = set(int(v) for v in np.unique(out_arr))
                all_colors = sorted(in_colors | out_colors)

                color_map: dict[int, int] = {}
                for r in range(min(h_in, h_out)):
                    for c in range(min(w_in, w_out)):
                        ic = int(in_arr[r, c])
                        oc = int(out_arr[r, c])
                        if ic != oc:
                            color_map[ic] = oc

                bbox_in: dict = {}
                bbox_out: dict = {}
                for col in in_colors:
                    boxes = self._bounding_boxes(in_arr, col)
                    if boxes:
                        bbox_in[str(col)] = boxes
                for col in out_colors:
                    boxes = self._bounding_boxes(out_arr, col)
                    if boxes:
                        bbox_out[str(col)] = boxes

                translations: dict = {}
                for col in all_colors:
                    vec = self._translation_vector(in_arr, out_arr, col)
                    if vec is not None:
                        translations[str(col)] = vec

                symmetry = self._symmetry_tests(in_arr, out_arr)

                histograms_in: dict = {}
                histograms_out: dict = {}
                for col in in_colors:
                    histograms_in[str(col)] = self._histograms(in_arr, col)
                for col in out_colors:
                    histograms_out[str(col)] = self._histograms(out_arr, col)

                if split == "train":
                    for col in range(10):
                        m_in = (in_arr == col).astype(np.uint8)
                        m_out = (out_arr == col).astype(np.uint8)

                        if col not in static_input_masks:
                            static_input_masks[col] = m_in
                            static_output_masks[col] = m_out
                            static_input_unchanged[col] = True
                            static_output_unchanged[col] = True
                        else:
                            prev_in = static_input_masks[col]
                            prev_out = static_output_masks[col]
                            if prev_in.shape != m_in.shape:
                                if prev_in.any() or m_in.any():
                                    static_input_unchanged[col] = False
                            elif not np.array_equal(prev_in, m_in):
                                static_input_unchanged[col] = False

                            if prev_out.shape != m_out.shape:
                                if prev_out.any() or m_out.any():
                                    static_output_unchanged[col] = False
                            elif not np.array_equal(prev_out, m_out):
                                static_output_unchanged[col] = False

                if invariant_input_colors is None:
                    invariant_input_colors = in_colors
                    invariant_output_colors = out_colors
                else:
                    invariant_input_colors &= in_colors
                    invariant_output_colors &= out_colors

                if h_in > 0 and h_out > 0:
                    shape_ratios.append((h_out / h_in, w_out / w_in))

                stats["examples"].append({
                    "split": split,
                    "input_shape": [h_in, w_in],
                    "output_shape": [h_out, w_out],
                    "input_grid": in_arr.tolist(),
                    "output_grid": out_arr.tolist(),
                    "input_colors": sorted(in_colors),
                    "output_colors": sorted(out_colors),
                    "color_mapping": {str(k): v for k, v in color_map.items()},
                    "new_colors_in_output": sorted(out_colors - in_colors),
                    "removed_colors_in_output": sorted(in_colors - out_colors),
                    "bounding_boxes_input": bbox_in,
                    "bounding_boxes_output": bbox_out,
                    "translation_vectors": translations,
                    "symmetry_transforms": symmetry,
                    "histograms_input": histograms_in,
                    "histograms_output": histograms_out,
                })

        static_input_channels = [
            col for col in range(10)
            if static_input_unchanged.get(col, False) and col in static_input_masks
        ]
        static_output_channels = [
            col for col in range(10)
            if static_output_unchanged.get(col, False) and col in static_output_masks
        ]

        stats["invariant_input_colors"] = sorted(invariant_input_colors or [])
        stats["invariant_output_colors"] = sorted(invariant_output_colors or [])
        stats["shapes_identical"] = all(r == (1.0, 1.0) for r in shape_ratios)
        stats["shape_ratios"] = shape_ratios
        stats["cross_example_static_input_channels"] = static_input_channels
        stats["cross_example_static_output_channels"] = static_output_channels
        stats["onnx_channel_note"] = (
            "ONNX tensor shape is [1,10,30,30]. Channel index == color index (0-9). "
            "One-hot: tensor[0][color][row][col] == 1.0 if that cell has that color."
        )

        return json.dumps(stats, indent=2)

class ExecuteOnnxCodeTool(BaseTool):
    name: str = "execute_onnx_code"
    description: str = (
        "Executes Python code that builds and saves an ONNX model. "
        "Input: python_code (str) — full Python source using onnx.helper that ends with "
        "onnx.save(model, output_path). "
        "Also input: output_path (str) — the path where the model should be saved. "
        "The code must contain a variable named output_path or use the provided path. "
        "Returns: SUCCESS message or EXECUTION_ERROR with stderr."
    )

    def _run(self, python_code: str, output_path: str = "") -> str:
        _SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
        script_path = _SCRATCH_DIR / "build_model.py"

        if output_path:
            python_code = re.sub(
                r'^\s*output_path\s*=.*$', '', python_code, flags=re.MULTILINE
            )
            preamble = f'output_path = {repr(output_path)}\n'
        else:
            preamble = ''

        full_code = preamble + python_code
        script_path.write_text(full_code)

        try:
            proc = subprocess.run(
                [sys.executable, str(script_path)],
                capture_output=True,
                text=True,
                timeout=60,
            )
        except subprocess.TimeoutExpired:
            return "EXECUTION_ERROR: Script timed out after 60 seconds."

        if proc.returncode != 0:
            stderr = proc.stderr[-3000:] if len(proc.stderr) > 3000 else proc.stderr
            return f"EXECUTION_ERROR:\n{stderr}"

        target = output_path or str(_SCRATCH_DIR / "model.onnx")
        if not check_network(target):
            return f"SIZE_ERROR: Model file at {target} failed size/existence check."

        return f"SUCCESS: model saved to {target}"

class AuditNetworkTool(BaseTool):
    name: str = "audit_network"
    description: str = (
        "Validates an ONNX model against ARC-AGI task examples and computes performance metrics. "
        "Input: model_path (str) — path to the .onnx file. "
        "Input: task_num (int) — task number (0 for the built-in test task). "
        "Returns: JSON string with pass/fail status, score, memory, params, and node names."
    )

    def _run(self, model_path: str, task_num: int = 0) -> str:
        result = {
            "passed": False,
            "arc_agi_right": 0,
            "arc_agi_wrong": 0,
            "arc_gen_right": 0,
            "arc_gen_wrong": 0,
            "memory_bytes": None,
            "params": None,
            "score": 1.0,
            "node_names": [],
            "errors": [],
        }

        if not check_network(model_path):
            result["errors"].append(f"File check failed for {model_path}")
            return json.dumps(result)

        try:
            sanitized = onnx.load(model_path)
        except Exception as e:
            result["errors"].append(f"onnx.load failed: {e}")
            return json.dumps(result)

        for node in sanitized.graph.node:
            if node.output:
                node.name = node.output[0]
            if "kernel_time" in node.name:
                result["errors"].append("Banned string 'kernel_time' found in node name.")
                return json.dumps(result)

        result["node_names"] = [n.name for n in sanitized.graph.node]

        try:
            examples = load_examples(task_num)
        except Exception as e:
            result["errors"].append(f"load_examples failed: {e}")
            return json.dumps(result)

        try:
            options = onnxruntime.SessionOptions()
            options.enable_profiling = True
            options.graph_optimization_level = (
                onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
            )
            unique_prefix = f"audit_{task_num:03d}_{int(time.time())}"
            options.profile_file_prefix = unique_prefix
            session = onnxruntime.InferenceSession(
                sanitized.SerializeToString(), options
            )
        except onnxruntime.ONNXRuntimeError as e:
            result["errors"].append(f"Session creation failed: {e}")
            return json.dumps(result)

        arc_agi_examples = examples.get("train", []) + examples.get("test", [])
        arc_gen_examples = examples.get("arc-gen", [])

        r, w, _ = verify_subset(session, arc_agi_examples)
        result["arc_agi_right"] = r
        result["arc_agi_wrong"] = w

        r2, w2, _ = verify_subset(session, arc_gen_examples)
        result["arc_gen_right"] = r2
        result["arc_gen_wrong"] = w2

        try:
            trace_path = session.end_profiling()
            memory, params = score_network(sanitized, trace_path)
            result["memory_bytes"] = memory
            result["params"] = params

            if memory is not None and params is not None and memory >= 0 and params >= 0:
                result["score"] = max(1.0, 25.0 - math.log(max(1.0, memory + params)))
        except Exception as e:
            result["errors"].append(f"score_network failed: {e}")

        total_wrong = result["arc_agi_wrong"] + result["arc_gen_wrong"]
        result["passed"] = total_wrong == 0

        return json.dumps(result, indent=2)

## 6. Agents

In [ ]:
from crewai import Agent


def build_llm(role: str) -> GemmaLocalLLM:
    temps = {"analyst": 0.3, "architect": 0.2, "auditor": 0.1}
    return GemmaLocalLLM(temperature=temps.get(role, 0.2))


def create_agents(
    llm_analyst: GemmaLocalLLM,
    llm_architect: GemmaLocalLLM,
    llm_auditor: GemmaLocalLLM,
) -> dict:
    analyst = Agent(
        role="ARC-AGI Pattern Recognition Specialist",
        goal=(
            "Analyze ARC-AGI puzzle grids and produce a concrete, actionable ONNX "
            "10-channel spatial masking strategy for the Architect to implement."
        ),
        backstory=(
            "You are a world-class expert in visual pattern recognition and tensor algebra. "
            "You deeply understand ARC-AGI grids as [1,10,30,30] FLOAT32 tensors where "
            "channel index equals color index (0-9). Each cell (row, col) of color C has "
            "tensor[0][C][row][col] == 1.0; all other channels at that position are 0. "
            "\n\n"
            "MANDATORY FIRST ACTION: call grid_stats with the integer task_num before any "
            "reasoning. Never skip this \u2014 it provides bounding boxes, translation vectors, "
            "symmetry flags, and cross-example invariants that ground your analysis in facts. "
            "\n\n"
            "ABSOLUTE PROHIBITION ON CELL ITERATION: You NEVER describe a solution in terms "
            "of loops, Python iteration, or cell-by-cell inspection. Phrases like 'for each "
            "row', 'iterate over cells', or 'check each pixel' are forbidden. Every thought "
            "MUST be expressed as a bulk tensor operation on the [1,10,30,30] spatial domain. "
            "Example of CORRECT reasoning: 'Channel 3 of the tensor is shifted 2 rows down "
            "via Pad([0,0,2,0,0,0,0,0]) followed by Slice([0,0,0,0,1,10,28,30], axes=[0,1,2,3]).' "
            "Example of FORBIDDEN reasoning: 'For each row, if color is 3, move it down 2.' "
            "\n\n"
            "You think exclusively in parameter-free ONNX ops: Slice, Gather, Transpose, "
            "Reshape, Where, Equal, Add, Sub, Mul, Pad, Concat, Squeeze, Unsqueeze. "
            "You NEVER suggest Conv, MatMul, or Gemm \u2014 they introduce trainable parameters "
            "that collapse the score. "
            "\n\n"
            "Your output is a numbered, concrete strategy: "
            "(1) which channels to select, "
            "(2) which spatial transformations to apply (shifts, crops, pads), "
            "(3) how to combine channels (Where, Equal, logical masking), "
            "(4) the final ONNX op sequence in order with intermediate tensor shapes."
        ),
        tools=[GridStatsTool()],
        llm=llm_analyst,
        allow_delegation=False,
        verbose=True,
        max_iter=10,
    )

    architect = Agent(
        role="ONNX Graph Engineer",
        goal=(
            "Translate the Analyst's strategy into executable Python code using onnx.helper "
            "that builds and saves a correct, minimal .onnx model, then run it with "
            "execute_onnx_code to confirm it executes without errors."
        ),
        backstory=(
            "You are a senior ML compiler engineer specializing in ONNX graph construction. "
            "You write production-quality Python \u2014 never pseudocode, never placeholder comments. "
            "\n\n"
            "STRICT RULES you never violate:\n"
            "1. IR version 10, opset 10: use onnx.helper.make_opsetid('', 10)\n"
            "2. Input tensor: name='input', dtype=FLOAT, shape=[1,10,30,30]\n"
            "3. Output tensor: name='output', dtype=FLOAT, shape=[1,10,30,30]\n"
            "4. BANNED ops: LOOP, SCAN, NONZERO, UNIQUE, SCRIPT, FUNCTION, COMPRESS \u2014 "
            "   any of these in the graph causes IMMEDIATE REJECTION by the scorer.\n"
            "5. STATIC SHAPES ONLY \u2014 no dynamic dims, no dim_param, no symbolic shapes. "
            "   A single dynamic dimension is AUTOMATIC DISQUALIFICATION from Neurogolf "
            "   scoring. Every intermediate tensor must have fully-resolved integer dims.\n"
            "6. Name EVERY node with a descriptive string, e.g. name='Slice_Color2_Rows'\n"
            "7. Single graph input ('input'), single graph output ('output')\n"
            "8. No tensor name may collide with any initializer name\n"
            "9. No tensor name may contain 'kernel_time'\n"
            "\n"
            "SCORING: score = max(1.0, 25.0 - ln(memory_bytes + params)). "
            "Favor ops that add ZERO parameters: Slice, Gather, Where, Equal, Transpose, "
            "Reshape, Pad, Concat, Add, Sub, Mul. Avoid Conv/MatMul/Gemm. "
            "\n\n"
            "After writing the code, call execute_onnx_code with the full code and "
            "the correct output_path. If execution fails, read the error, fix the code, "
            "and try again. You may attempt fixes up to 4 times."
        ),
        tools=[ExecuteOnnxCodeTool()],
        llm=llm_architect,
        allow_delegation=False,
        verbose=True,
        max_iter=15,
    )

    auditor = Agent(
        role="ONNX Efficiency Principal Engineer",
        goal=(
            "Validate the generated ONNX model using audit_network, then produce "
            "structured, metric-based engineering feedback that guides the next attempt."
        ),
        backstory=(
            "You are a principal engineer who audits ONNX graphs for correctness and efficiency. "
            "You call audit_network and receive a JSON result with: pass/fail status, "
            "right/wrong example counts, memory_bytes, params, score, and node_names. "
            "\n\n"
            "You reason over these metrics to write actionable feedback. "
            "You ALWAYS include 'score: X.XX' somewhere in your output (the exact float). "
            "You ALWAYS start your report with 'PASS' or 'FAIL' on the first line. "
            "\n\n"
            "Your feedback structure:\n"
            "- Status: PASS or FAIL\n"
            "- Score: the numeric score from the audit\n"
            "- Accuracy: how many examples passed/failed\n"
            "- If FAIL or score < 15.0: identify specific nodes by name from node_names "
            "  that are causing problems, and suggest STRATEGY-level fixes "
            "  (e.g., 'Node Slice_Color2 outputs wrong shape; the Slice axes parameter "
            "  should be [0,1] not [1,2]'). "
            "\n\n"
            "CRITICAL: You do NOT write Python code. You give engineering guidance only. "
            "Reference node names exactly as they appear in the audit JSON node_names list."
        ),
        tools=[AuditNetworkTool()],
        llm=llm_auditor,
        allow_delegation=False,
        verbose=True,
        max_iter=5,
    )

    return {
        "analyst": analyst,
        "architect": architect,
        "auditor": auditor,
    }


## 7. Tasks

In [ ]:
from crewai import Task


def create_tasks(
    agents: dict,
    task_num: int,
    attempt: int,
    model_output_path: str,
    feedback: str,
    best_score: float,
) -> list:
    """
    Create a fresh set of 3 Tasks for one attempt at solving a given ARC-AGI puzzle.

    New Task objects are created each call to avoid CrewAI caching stale descriptions.
    The examples JSON is NOT embedded in templates — tools load it directly by task_num
    to avoid Python str.format_map conflicts with JSON curly braces.
    """
    feedback_section = feedback if feedback else "No previous feedback — this is attempt 1."
    best_score_str = f"{best_score:.3f}"

    analysis_task = Task(
        description=(
            f"You are analyzing ARC-AGI task #{task_num}, attempt {attempt}/10.\n\n"
            f"Step 1: Call the grid_stats tool with task_num={task_num}. "
            f"Do NOT attempt to read files yourself — the tool loads the puzzle internally. "
            f"Your ONLY input to the tool is the integer {task_num}.\n\n"
            f"Step 2: Using the stats output, perform Chain-of-Thought reasoning to identify:\n"
            f"  (a) What spatial transformation maps input grids to output grids?\n"
            f"  (b) Which color channels (0-9) are involved and how?\n"
            f"  (c) Is it a shift, rotation, masking, color replacement, or spatial selection?\n"
            f"  (d) Which ONNX ops implement this with ZERO trainable parameters?\n\n"
            f"CRITICAL: Express ALL reasoning as bulk tensor ops on [1,10,30,30] channel-space. "
            f"NEVER describe a solution using loops, iteration, or cell-by-cell language.\n\n"
            f"Step 3: Produce a numbered strategy listing the exact op sequence with "
            f"intermediate tensor shapes at every step.\n\n"
            f"Previous attempt feedback:\n{feedback_section}"
        ),
        expected_output=(
            "A clear, numbered strategy in natural language describing: "
            "(1) the transformation type, "
            "(2) which channels (0-9) to operate on, "
            "(3) the specific ONNX ops to use (Slice/Gather/Where/Equal/Pad/etc.), "
            "(4) the complete op sequence with intermediate tensor shapes at each step."
        ),
        agent=agents["analyst"],
    )

    architecture_task = Task(
        description=(
            f"You are implementing ARC-AGI task #{task_num}, attempt {attempt}/10 as an ONNX model.\n\n"
            f"Save the model to: {model_output_path}\n\n"
            f"The Analyst has given you a strategy (see context above). Implement it as "
            f"complete, executable Python code using onnx.helper. Requirements:\n"
            f"  - import onnx; from onnx import helper, TensorProto\n"
            f"  - IR version 10: onnx.helper.make_opsetid('', 10)\n"
            f"  - Input: make_tensor_value_info('input', TensorProto.FLOAT, [1,10,30,30])\n"
            f"  - Output: make_tensor_value_info('output', TensorProto.FLOAT, [1,10,30,30])\n"
            f"  - Name every node descriptively, e.g. name='Slice_Extract_Color3'\n"
            f"  - BANNED ops (IMMEDIATE REJECTION): LOOP, SCAN, NONZERO, UNIQUE, SCRIPT, "
            f"FUNCTION, COMPRESS\n"
            f"  - STATIC SHAPES ONLY: every intermediate tensor must have fully-resolved "
            f"integer dims. ANY dynamic dim or dim_param is AUTOMATIC DISQUALIFICATION.\n"
            f"  - End with: onnx.save(model, output_path)  # output_path is pre-set for you\n\n"
            f"After writing the code, call execute_onnx_code with:\n"
            f"  - python_code = <your full code>\n"
            f"  - output_path = {repr(model_output_path)}\n\n"
            f"If execution returns EXECUTION_ERROR, read the error, fix the code, and retry "
            f"(up to 4 times).\n\n"
            f"Auditor feedback from previous attempt:\n{feedback_section}\n"
            f"Best score so far: {best_score_str}"
        ),
        expected_output=(
            f"Confirmation that execute_onnx_code returned SUCCESS and the model is saved "
            f"at {model_output_path}. If all retries failed, report the final error."
        ),
        agent=agents["architect"],
        context=[analysis_task],
    )

    audit_task = Task(
        description=(
            f"You are auditing the ONNX model for ARC-AGI task #{task_num}, attempt {attempt}/10.\n\n"
            f"Call audit_network with:\n"
            f"  - model_path = {repr(model_output_path)}\n"
            f"  - task_num = {task_num}\n\n"
            f"Analyze the JSON result. Your report MUST:\n"
            f"  1. Start with 'PASS' or 'FAIL' on the first line\n"
            f"  2. Include 'score: X.XX' with the exact float score\n"
            f"  3. Report arc_agi_right/wrong and arc_gen_right/wrong counts\n"
            f"  4. If FAIL: identify which examples failed and why (reference node names "
            f"     from the node_names list)\n"
            f"  5. If score < 15.0: name specific nodes driving parameter/memory cost "
            f"     and suggest a strategy-level fix\n\n"
            f"Do NOT write Python code. Write engineering guidance only.\n"
            f"Target score: >= 15.0"
        ),
        expected_output=(
            "A structured audit report containing: "
            "PASS/FAIL on line 1, "
            "'score: X.XX' on line 2, "
            "example accuracy counts, "
            "and (if needed) specific node-level optimization suggestions referencing "
            "exact node names from the audit JSON."
        ),
        agent=agents["auditor"],
        context=[architecture_task],
    )

    return [analysis_task, architecture_task, audit_task]


## 8. Main orchestration

In [ ]:
import re
import sys
from pathlib import Path

from crewai import Crew, Process

OUTPUT_DIR = Path("outputs")
NUM_TASKS = 1
MAX_ATTEMPTS = 3
TARGET_SCORE = 20.0

def parse_score_from_audit(text: str) -> float:
    """Extract the float score from the Auditor's output text."""
    match = re.search(r"score[:\s]+([0-9]+\.?[0-9]*)", text, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            pass
    return 0.0

def parse_pass_from_audit(text: str) -> bool:
    """Check if the Auditor's output indicates all examples passed."""
    first_line = text.strip().splitlines()[0].upper() if text.strip() else ""
    if "PASS" in first_line:
        return True
    lower = text.lower()
    return (
        "all examples pass" in lower
        or "network is ready" in lower
        or ("pass" in lower and "fail" not in lower[:200])
    )

def solve_task(task_num: int, agents: dict) -> tuple:
    """
    Run up to MAX_ATTEMPTS to solve a single ARC-AGI task.
    Returns (best_score, best_model_path).
    """
    best_score = 0.0
    best_model_path = None
    feedback = ""

    for attempt in range(1, MAX_ATTEMPTS + 1):
        model_path = OUTPUT_DIR / f"task_{task_num:03d}_attempt_{attempt:02d}.onnx"
        print(f"\n  [Task {task_num} | Attempt {attempt}/{MAX_ATTEMPTS}] "
              f"Target: {model_path.name}")

        tasks = create_tasks(
            agents=agents,
            task_num=task_num,
            attempt=attempt,
            model_output_path=str(model_path),
            feedback=feedback,
            best_score=best_score,
        )

        crew = Crew(
            agents=list(agents.values()),
            tasks=tasks,
            process=Process.sequential,
            verbose=True,
        )

        try:
            result = crew.kickoff()

            # Extract Auditor's output (last task)
            audit_text = ""
            if result.tasks_output:
                audit_text = result.tasks_output[-1].raw or ""

            score = parse_score_from_audit(audit_text)
            passed = parse_pass_from_audit(audit_text)

            print(f"  Passed={passed} | Score={score:.3f}")

            if passed and score > best_score:
                best_score = score
                best_model_path = str(model_path)

            if passed and score >= TARGET_SCORE:
                print(f"  Target reached ({score:.3f} >= {TARGET_SCORE}). Moving on.")
                break

            feedback = audit_text

        except Exception as e:
            print(f"  [ERROR] Crew failed on attempt {attempt}: {e}")
            feedback = f"Previous attempt {attempt} failed with system error: {e}"

    return best_score, best_model_path

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    llm_analyst = build_llm("analyst")
    llm_architect = build_llm("architect")
    llm_auditor = build_llm("auditor")

    agents = create_agents(llm_analyst, llm_architect, llm_auditor)

    results: dict[int, tuple] = {}

    for task_num in range(1, NUM_TASKS + 1):
        print(f"\n{'=' * 60}")
        print(f"Solving Task {task_num}/{NUM_TASKS}")
        print(f"{'=' * 60}")

        try:
            _load_task(task_num)
        except Exception as e:
            print(f"  [SKIP] Task {task_num}: JSON not found — {e}")
            results[task_num] = (0.0, None)
            continue

        score, model_path = solve_task(task_num, agents)
        results[task_num] = (score, model_path)
        print(f"Task {task_num} complete — best score: {score:.3f} | model: {model_path}")

    print(f"\n{'=' * 60}")
    print("FINAL RESULTS")
    print(f"{'=' * 60}")
    total_score = sum(s for s, _ in results.values())
    solved = sum(1 for s, _ in results.values() if s >= TARGET_SCORE)
    print(f"Total score: {total_score:.3f}")
    print(f"Tasks reaching target (>= {TARGET_SCORE}): {solved}/{NUM_TASKS}")
    for tn, (sc, mp) in sorted(results.items()):
        status = "OK" if sc >= TARGET_SCORE else "  "
        print(f"  [{status}] Task {tn:03d}: {sc:.3f}  {mp or 'no model'}")


## 9. Run

In [ ]:
main()
